# Descriptive Alerts — Qwen2-VL-2B LoRA on Charades

Fine-tunes `Qwen/Qwen2-VL-2B-Instruct` with LoRA to emit one-line activity alerts
("A person is closing a door.") from short home-video clips — the Wyze *Descriptive Alerts* task.

**Runtime:** Kaggle, GPU T4 x2 (only one GPU is used; a 2B LoRA does not need model parallelism).

**Before running:** set accelerator to GPU, attach your private frames dataset,
and add an `HF_TOKEN` secret if you want to push the adapter at the end.

Every expensive cell is **skip-if-exists**, so re-running after a session timeout
resumes instead of recomputing. Order is deliberate: the headline comparison
finishes first, the secondary prompt arms last.


## 0. Environment check


In [ ]:
!nvidia-smi
import torch, platform
print('torch', torch.__version__, '| cuda', torch.version.cuda)
print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
print('compute capability', cap)
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
# T4 is SM 7.5: no bf16, no FlashAttention-2. The configs default to fp16 + sdpa.
assert torch.cuda.is_available(), 'Enable the GPU accelerator in notebook settings.'


## 1. Dependencies

Kaggle ships torch with CUDA already — do **not** reinstall torch, it breaks the CUDA build.
Versions are pinned to what the pipeline was validated against.


In [ ]:
!pip install -q --no-deps transformers==5.19.0 peft==0.21.2
!pip install -q accelerate av pyyaml pycocoevalcap
import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)


## 2. Code + data

Frames come from the attached private Kaggle dataset (see `scripts/05_pack_for_kaggle.py`).
Charades is non-commercial research data: keep that dataset private.


In [ ]:
REPO = 'https://github.com/ksujith/descriptive-alerts-vlm'
import os, pathlib, subprocess
if not pathlib.Path('/kaggle/working/descriptive-alerts-vlm').exists():
    !git clone -q $REPO /kaggle/working/descriptive-alerts-vlm
%cd /kaggle/working/descriptive-alerts-vlm
!git log --oneline -1


In [ ]:
# Point this at your attached dataset directory.
DATA = '/kaggle/input/charades-alert-frames'
import pathlib, tarfile, time
assert pathlib.Path(DATA).exists(), f'attach the frames dataset at {DATA}'
print(sorted(p.name for p in pathlib.Path(DATA).iterdir()))


In [ ]:
# Extract to local SSD (fast); skip if already done.
import tarfile, pathlib, time
root = pathlib.Path('/kaggle/working/descriptive-alerts-vlm')
for tar in sorted(pathlib.Path(DATA).glob('*.tar')):
    marker = root / f'.extracted_{tar.stem}'
    if marker.exists():
        print('skip', tar.name); continue
    t0 = time.time()
    with tarfile.open(tar) as tf: tf.extractall(root)
    marker.touch()
    print(f'extracted {tar.name} in {time.time()-t0:.0f}s')
# meta.tar lands in meta/; move the manifest and annotations where the code expects them.
!mkdir -p data/interim data/raw
!cp -n meta/clips.csv data/interim/clips.csv 2>/dev/null || true
!cp -rn meta/Charades data/raw/ 2>/dev/null || true
!ls data/interim data/raw data/frames 2>/dev/null | head -20
!find data/frames -name '*.jpg' | wc -l


## 3. Smoke test — prove the loop runs before spending quota

16 samples, one epoch. If this completes, shapes, label masking and the fp16
LoRA setup are all sound.


In [ ]:
!python -m dalerts.train --config configs/smoke.yaml 2>&1 | tail -30


## 4. Train (8 frames)

Resumable: re-running picks up the newest checkpoint automatically.
Expect roughly 4-8 h for 2 epochs over ~14k clips on one T4.


In [ ]:
!PYTHONPATH=src python -m dalerts.train --config configs/t4_8frame.yaml --resume auto 2>&1 | tail -40


In [ ]:
!ls -la runs/t4_8frame/adapter
!du -sh runs/t4_8frame/adapter


## 5. Evaluation

Headline comparison first (baseline vs fine-tuned, identical prompt and frames),
then the frame-count ablation, then the secondary baseline prompt arms.
Each cell writes its own JSONL and is skipped if that file already exists.


In [ ]:
ADAPTER = 'runs/t4_8frame/adapter'
COMMON = '--manifest data/interim/clips.csv --frames data/frames --charades data/raw/Charades --split test --batch-size 4'
print(COMMON)


In [ ]:
# --- headline pair ---
!python scripts/03_generate.py $COMMON --arm formatted --n-frames 8 --tag baseline
!python scripts/03_generate.py $COMMON --arm formatted --n-frames 8 --tag ft --adapter $ADAPTER


In [ ]:
# --- frame-count ablation (same adapter, strided from the same cached frames) ---
for n in [16, 4, 1]:
    !python scripts/03_generate.py $COMMON --arm formatted --n-frames {n} --tag ft --adapter $ADAPTER


In [ ]:
# --- secondary baseline arms: how much of the gap is just output format? ---
for arm in ['open', 'fewshot', 'choice']:
    !python scripts/03_generate.py $COMMON --arm {arm} --n-frames 8 --tag baseline


## 6. Report


In [ ]:
!python scripts/04_report.py --preds reports/preds --charades data/raw/Charades
from IPython.display import Markdown, display
display(Markdown(open('reports/RESULTS.md').read()))


## 7. Push the adapter to the Hub

Add your token as a Kaggle secret named `HF_TOKEN` (Add-ons -> Secrets).
Only the LoRA adapter, config and model card are uploaded — **never the frames**.


In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
REPO_ID = 'YOUR_HF_USERNAME/qwen2vl-2b-charades-alerts-lora'  # <- edit
!python scripts/06_push_to_hf.py --adapter runs/t4_8frame/adapter --repo-id $REPO_ID \
    --results reports/results.json --run-config runs/t4_8frame/config.json


In [ ]:
# Keep the artifacts if the session ends.
!tar cf /kaggle/working/results.tar reports runs/t4_8frame/config.json runs/t4_8frame/log_history.json
!ls -la /kaggle/working/results.tar
